In [26]:
import torch
from torch import nn
from torch import Tensor

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.mps.is_available() else 'cpu')

class ManualModel:
    w1: Tensor = None
    w2: Tensor = None
    w3: Tensor = None
    b1: Tensor = None
    b2: Tensor = None
    b3: Tensor = None

    def __init__(self, we1, we2, we3, bi1, bi2, bi3):
        self.w1 = we1
        self.w2 = we2
        self.w3 = we3
        self.b1 = bi1
        self.b2 = bi2
        self.b3 = bi3

    def printModelWeightsBiases(self):
        print("*" * 80)
        print(f'\n\nw1={self.w1}, \n\nb1={self.b1}, \n\nw2={self.w2}, \n\nb2={self.b2}, \n\nw3={self.w3}, \n\nb3={self.b3}\n\n')


# from torch import Tensor
# t1 = Tensor([[1.0, 2.0, 3.0], 
#              [4.0, 5.0, 6.0]])

# print(f'sum={t1.sum(dim=0)}')


In [27]:
# create dataset, dataloader -> data cleaning, formatting etc.
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset

mnist_train = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), target_transform=None, download=False)
mnist_test = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), target_transform=None, download=False)

# print(mnist_train.data.shape)

# for i in range(3):
#     feature, label = mnist_train[i]
#     print(f"Sample {i}:")
#     print(f"  Feature shape: {feature.shape}")
#     print(f"  Label: {label}")

# print(mnist_train.targets[:3])
# assert(False)

# flattened_mnist_train = torch.flatten(input=mnist_train.data, start_dim=1)
# flattened_mnist_train_64 = flattened_mnist_train[:64].float().to(device=DEVICE)

# print(f'flattened_mnist_train_64={flattened_mnist_train_64}')

# flattened_mnist_train_64_labels = mnist_train.targets[:64].to(device=DEVICE) # shape = [64]


# print(f'labels_one_hot.shape: {labels_one_hot.shape}')
# print(labels_one_hot[:, :3])

# print(flattened_mnist_train.data.shape)

# Create a DataLoader
MANUAL_LEARNING_RATE = 0.1
BATCH_SIZE = 64
EPOCHS = 5

train_loader = DataLoader(dataset=mnist_train, batch_size=BATCH_SIZE)
test_loader = DataLoader(dataset=mnist_test, batch_size=BATCH_SIZE)


In [28]:
# create the model = nn.Sequential (pick a model architecture)
INPUT_SIZE = 28 * 28
HIDDEN_SIZE = 128
OUTPUT_SIZE = 10

seq_model = nn.Sequential(
                      nn.Flatten(),
                      nn.Linear(in_features=INPUT_SIZE, out_features=HIDDEN_SIZE),
                      nn.ReLU(),
                      nn.Linear(in_features=HIDDEN_SIZE, out_features=HIDDEN_SIZE),
                      nn.ReLU(),
                      nn.Linear(in_features=HIDDEN_SIZE, out_features=OUTPUT_SIZE)
                      ).to(device=DEVICE)

# To match the auto loss values, initialize the weights the same as how auto does.
# Auto does it using He / Kaiming Uniform initialization internally.

# w1 = seq_model[1].weight.detach().clone().T
# b1 = seq_model[1].bias.detach().clone()
# w2 = seq_model[3].weight.detach().clone().T
# b2 = seq_model[3].bias.detach().clone()
# w3 = seq_model[5].weight.detach().clone().T
# b3 = seq_model[5].bias.detach().clone()

# Multiple weights by (1 / torch.sqrt(HIDDEN_SIZE) to keep the variance less
w1 = (torch.randn(size=(INPUT_SIZE, HIDDEN_SIZE)) * (1 / HIDDEN_SIZE**0.5)).to(DEVICE)
b1 = torch.zeros(HIDDEN_SIZE).to(DEVICE)
# print(f'w1[:3]: {w1[:3]}')
print(f'w1.shape: {w1.shape}')



w1.shape: torch.Size([784, 128])


In [29]:
from torch.nn import functional as F
from torch.optim import Adam
import torch.optim as optim
from tqdm import tqdm
# from matplotlib import pyplot

w2 = (torch.randn(size=(HIDDEN_SIZE, HIDDEN_SIZE)) * (1 / HIDDEN_SIZE**0.5)).to(DEVICE)
b2 = torch.zeros(HIDDEN_SIZE).to(DEVICE)

w3 = (torch.randn(size=(HIDDEN_SIZE, OUTPUT_SIZE)) * (1 / HIDDEN_SIZE**0.5)).to(DEVICE) # shape [128, 10]
b3 = torch.zeros(OUTPUT_SIZE).to(DEVICE)

man_model = ManualModel(w1, w2, w3, b1, b2, b3)

def train_one_epoch_manual(mdl: ManualModel, loader: DataLoader):

    total_loss = 0
    total_samples_processed = 0
    correct_predictions = 0

    for x, label in tqdm(iterable=loader):
        N = x.size(dim=0)

        # print(N)
        # print(label)
        # # plot the image
        # m = x[0:1, :, :]
        # print(f'x.shape: {m.shape}')
        # pyplot.imshow(m.squeeze())
        # pyplot.show()
        
        x, label = x.to(DEVICE), label.to(DEVICE)
        x = x.squeeze()

        x = torch.flatten(input=x, start_dim=1, end_dim=2)

        # MANUAL
        
        # 1st linear layer
        # Multiply input [64, 784] with the weights [784, 128]  to get y1 = [64, 128]
        y1 = x @ mdl.w1 + mdl.b1

        # then pass through relu, output shape = [64, 128]
        z1 = torch.relu(y1)

        # 2nd linear layer. Multiply z1 of shape [64, 128] with w2 of shape [128, 128] weights to get [64, 128] output
        y2 = z1 @ mdl.w2 + mdl.b2

        # then pass through relu, output shape = [64, 128]
        z2 = torch.relu(y2)

        # do I need a linear layer here? - Yes, for output of 10 neurons.
        # Multiply z2 of shape [64, 128] with w3 of shape [128, 10] to get output of size [64, 10]
        logits = z2 @ mdl.w3 + mdl.b3

        # Now do : loss = F.cross_entropy(logits, flattened_mnist_train_64_labels)

        # after passing it through the 2 layers + 1 linear layer, do a softmax to get a probability distribution of the multi-class classification
        # output size should be [64, 10]
        probs = torch.softmax(logits, dim=1) # normalized probabilities
        
        # convert the labels to one-hot representation, size = [10, 64]
        
        # Create a column vector of indices, size = [N, 1]
        idx = label.unsqueeze(dim=1)

        # print(f'idx.shape: {idx.shape}')
        labels_one_hot = torch.zeros(size=(N,OUTPUT_SIZE)).to(device=DEVICE)

        # Scatter horizontally (dim=1) across the classes
        labels_one_hot = labels_one_hot.scatter_(dim=1, index=idx, value=1.0)

        # print(f'probs={probs.shape}, labels_one_hot={labels_one_hot.shape}')

        # calculate the cross entropy loss without using F.cross_entropy
        # L = -log (Sigma { one-hot-vector-of-true-label * probabilities })
        loss = -(labels_one_hot * torch.log(probs + 1e-9)).sum(dim=0) # output should be [1,64]
        mean_loss = loss.mean(dim=0)
        total_loss += mean_loss
        total_samples_processed += N
        # print(f'mean_loss={mean_loss}')

        # BACKWARD PASS
        # - d loss / d w3 = (d loss / d probs) * (d probs / d logits) * (d logits / d w3) = CE loss's derivative * softmax's derivative * z2's value
        # -  CE loss's derivative * softmax's derivative = (probabilities - one hot vector) / num samples. 
        # This outputs a tensor of shape [64, 10] (i.e., shape of probs). z2 shape is [64, 128]. Gradient shape should be [128, 10] for it to match weights.
        d3 = ((probs - labels_one_hot) / N) # shape [64, 10]
        grad_w3 = z2.T @ d3

        # - d loss / d b3 = (d loss / d probs) * (d probs / d logits) * (d logits / d b3) = CE loss's derivative * softmax's derivative * 1
        # Gradient shape should be [10], i.e., all rows of d3 have to collapse
        grad_b3 = d3.sum(dim=0)

        # grad_w2 = (d loss / d probs) * (d probs / d logits) * (d logits / d z2) * (d z2 / d y2) * (d y2 / d w2)
        #         = d3 * w3 * relu's derivative * z1
        #         = [64,10] * [128, 10] * [64, 128] * [64, 128]
        d2 = (d3 @ mdl.w3.T) * (z2 > 0).float() # shape [64,128]
        grad_w2 = z1.T @ d2
        grad_b2 = d2.sum(dim=0)

        d1 = (d2 @ mdl.w2.T) * (z1 > 0).float()
        grad_w1 = x.T @ d1
        grad_b1 = d1.sum(dim=0)

        # Adjust the weights

        lr = MANUAL_LEARNING_RATE # learning rate
        mdl.w3 -= (lr * grad_w3) # shape [128, 10]
        mdl.b3 -= (lr * grad_b3) # shape [10]

        mdl.w2 -= (lr * grad_w2)
        mdl.b2 -= (lr * grad_b2)

        mdl.w1 -= (lr * grad_w1)
        mdl.b1 -= (lr * grad_b1)

        # print(f'w1={w1}, b1={b1}, w2={w2}, b2={b2}, w3={w3}, b3={b3}')
        
        correct_predictions += (probs.argmax(dim=1) == label).long().sum().item()

    return (total_loss / total_samples_processed, correct_predictions / total_samples_processed)



In [30]:
from tqdm.auto import tqdm
from torch.nn import functional as F

def train_one_epoch(model: nn.Sequential, optimizer: Adam, data_loader: DataLoader):
    
    # put the model in training mode
    model.train()

    total_loss = 0.0
    total_samples_processed = 0
    correct_predictions = 0

    # iterate over (mini)batches of data from the data loader and train the model for this 1 epoch
    for x, label in tqdm(iterable=data_loader):

        # print(f'x shape: {x.shape}, label shape: {label.shape}')

        # move the data to device
        x, label = x.to(DEVICE), label.to(DEVICE)

        # Infer from the model (forward pass).
        # Logits are unnormalized scores for the predicted classes (numbers from 1-10 in this case)
        #   Note: If you wanted actual probabilities (e.g., "73% confident it's a 3"), 
        #         you'd apply softmax to convert logits into a proper probability distribution over the 10 classes.
        # x.shape is (64, 1, 28, 28)
        logits = model(x)

        # check the loss between the model output and the label. F.cross_entropy returns the mean loss over the batch.
        cross_ent_loss: Tensor = F.cross_entropy(input=logits, target=label)
        # print(f'cross_ent_loss: {cross_ent_loss}')

        # Zero out of the (computed) gradients of all tensors because gradients are accumulated into the tensors
        optimizer.zero_grad()

        # # Compute the gradients of the loss w.r.t. the weights (leaves of the computation graph) using the chain rule.
        cross_ent_loss.backward()
        # This is the step that we need to do manually
        #   -> gradient of error w.r.t weights
        #   -> 
        

        # While loss.backward() only computes and stores the gradients in the Tensor's `.grad`, 
        # it doesn't touch the weights of the model. The following line actually does the
        # weight updates, thereby minimizing the loss.
        optimizer.step()

        ## Bookkeeping of loss/accuracy stats

        # print(f'cross_ent_loss: {cross_ent_loss},\
        #       cross_ent_loss.item(): {cross_ent_loss.item()},\
        #       x.size(0): {x.size(0)}')

        # Need to find the total loss of all samples in the batch (x.size(0) should be == BATCH_SIZE), across all mini-batches
        # .item() converts to standard python number
        total_loss += cross_ent_loss.item() * x.size(0)
        total_samples_processed += x.size(0)

        # Get the predictions (numbers from 1-10. logits is the unnormalized score tensor).
        # This is achieved by finding out the index of the maximum value in the logits tensor, which is of shape .
        prediction = logits.argmax(dim=1)
        # print(f'prediction.shape: {prediction.shape}')
        # print(f'preds: {prediction}')

        correct_predictions += (prediction == label).long().sum().item()

    # Return the tuple containing average loss and accuracy
    return (total_loss / total_samples_processed, correct_predictions / total_samples_processed)

# Create the optimizer (with default hyperparameters - learning rate = 0.001). 
# This links the optimizer to the model's parameters
adam_opt = Adam(params=seq_model.parameters())
sgd_opt = optim.SGD(seq_model.parameters(), lr=0.001)

# # Loop over the dataset in batches and train the model

# man_model.printModelWeightsBiases()

for epoch in range(0, EPOCHS):
    (avg_loss, avg_accuracy) = train_one_epoch_manual(mdl=man_model, loader=train_loader)
    print(f'avg_loss_man: {avg_loss}, avg_accuracy_man: {avg_accuracy}')

# # Print names and tensors for weights and biases
# print("*" * 80)
# for name, param in seq_model.named_parameters():
#     print(f"Layer: {name}")
#     print(param.data)
#     print("-" * 80)

# for epoch in range(0, EPOCHS):
#     (avg_loss, avg_accuracy) = train_one_epoch(model=seq_model, optimizer=adam_opt, data_loader=train_loader)
#     print(f'avg_loss: {avg_loss}, avg_accuracy: {avg_accuracy}')


100%|██████████| 938/938 [00:03<00:00, 303.62it/s]


avg_loss_man: 0.03630535304546356, avg_accuracy_man: 0.8950166666666667


100%|██████████| 938/938 [00:02<00:00, 354.91it/s]


avg_loss_man: 0.0166030116379261, avg_accuracy_man: 0.9505666666666667


100%|██████████| 938/938 [00:02<00:00, 351.67it/s]


avg_loss_man: 0.012237715534865856, avg_accuracy_man: 0.9641333333333333


100%|██████████| 938/938 [00:02<00:00, 355.11it/s]


avg_loss_man: 0.009656397625803947, avg_accuracy_man: 0.9714833333333334


100%|██████████| 938/938 [00:02<00:00, 353.02it/s]

avg_loss_man: 0.00786580890417099, avg_accuracy_man: 0.97705
